# Jour 1 · Mini-projet : choisir un modèle honnêtement

> Version corrigée — réponses et raisonnement après les exercices.

## Objectifs

- comparer plusieurs modèles avec le même protocole
- choisir sur la validation puis évaluer une seule fois sur le test
- formuler une recommandation compréhensible par le métier


## Mission

Vous devez proposer un premier détecteur de risque de panne. Trois candidats sont disponibles : régression logistique, arbre de décision et forêt aléatoire. Nous utiliserons la validation pour comparer les candidats. Le test ne sera ouvert qu'après le choix.

### Les trois candidats

- La **régression logistique** construit une frontière simple et reste facile à expliquer.
- Un **arbre de décision** enchaîne des questions du type « vibration supérieure à une valeur ? ». `max_depth=4` limite ici le nombre de niveaux pour éviter un arbre trop complexe.
- Une **forêt aléatoire** combine plusieurs arbres légèrement différents puis agrège leurs décisions. `n_estimators=150` demande 150 arbres ; `n_jobs=-1` autorise l'utilisation des cœurs CPU disponibles pour accélérer le calcul.

Ces paramètres sont des **hyperparamètres** : ils sont choisis avant l'entraînement, contrairement aux paramètres appris automatiquement par le modèle.



![Comparaison intuitive de trois familles de modèles](../../ressources/illustrations/jour_01/04_trois_modeles.png)

La régression logistique combine toutes les features dans un score puis le transforme en probabilité. Sa frontière de décision reste relativement simple. Les coefficients peuvent aider à comprendre le sens d’une relation, à condition que les données aient été correctement préparées.

L’arbre de décision découpe progressivement les observations. Chaque nœud pose une question, chaque branche représente une réponse et chaque feuille produit une décision. Un arbre profond peut créer des règles très spécifiques aux données d’entraînement.

La forêt aléatoire entraîne de nombreux arbres sur des échantillons et des sous-ensembles de features légèrement différents. Le vote de plusieurs modèles réduit souvent l’instabilité d’un arbre isolé. En contrepartie, expliquer chaque décision devient moins direct.

Aucune famille n’est toujours gagnante. Nous devons les comparer avec le même découpage, la même cible et les mêmes métriques.


![Comparaison des modèles avec le même protocole](../../ressources/illustrations/jour_01/04_comparer_modeles.png)

*Une comparaison honnête conserve exactement les mêmes données et métriques pour tous les candidats.*


### Le fil de votre mini-projet

![Étapes du mini-projet et place du test](../../ressources/illustrations/jour_01/04_mission_progressive.png)

Vous allez compléter quatre décisions : entraîner et comparer, choisir, autoriser le contrôle final, puis expliquer une force et une limite. Les parties incomplètes affichent un message et restent exécutables. Pour avancer, complétez la première étape manquante avant d'exécuter les suivantes.


### Préparer les outils, puis ouvrir le tableau

`import` rend disponible une bibliothèque, c'est-à-dire un ensemble d'outils déjà écrits. `as pd`, par exemple, lui donne un nom court. **pandas** lit et manipule les tableaux ; **NumPy** calcule avec des tableaux de nombres ; **Matplotlib** dessine. Une **variable** est un nom associé à une valeur : `df` désignera notre tableau.

Exécutez les cellules de haut en bas, avec **Maj + Entrée**. Une cellule terminée peut produire un texte, un tableau ou un graphique ; l'absence de sortie n'est pas une erreur.


In [ ]:
from pathlib import Path  # Path décrit un chemin de fichier, sur Windows comme sur Linux.
import numpy as np  # np désigne les outils de calcul numérique.
import pandas as pd  # pd désigne les outils de tableaux.
import matplotlib.pyplot as plt  # plt désigne les outils de dessin.
from IPython.display import display  # display affiche joliment un tableau dans Jupyter.

plt.style.use("seaborn-v0_8-whitegrid")  # Ajoute une grille lisible aux graphiques.


Le petit outil suivant retrouve le dossier du cours. `def` définit une **fonction**, un bloc réutilisable ; `for` essaie plusieurs dossiers ; `if` teste une condition ; `return` fournit le résultat. `Path.cwd()` est le dossier courant et `.parents` ses dossiers parents. Le `/` entre chemins assemble les noms, il ne fait pas une division.

C'est une étape technique de chargement, pas une méthode de Machine Learning. Vous pouvez exécuter cette fonction telle quelle.


In [ ]:
def find_course_root():
    """Chercher le dossier qui contient les données du cours."""
    current = Path.cwd()  # Dossier depuis lequel Jupyter exécute ce notebook.
    for candidate in (current, *current.parents):  # * ajoute les parents à la liste d'essais.
        data_path = candidate / "ressources" / "donnees" / "maintenance_machines.csv"
        if data_path.exists():  # Arrêter la recherche dès que le fichier est trouvé.
            return candidate
    raise FileNotFoundError("Ouvrez le notebook depuis le dossier fondamentaux_ml_dl.")

ROOT = find_course_root()  # ROOT garde le chemin du dossier contenant les données.
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"


In [ ]:
df = pd.read_csv(DATA_FILE)  # Lire le CSV : df est maintenant un DataFrame (tableau).
display(df.head(3))  # Montrer 3 lignes seulement ; le tableau complet n'est pas modifié.


### Retrouver les entrées et la réponse

Une liste Python utilise des crochets : `["a", "b"]`. `df[features]` sélectionne plusieurs colonnes et garde un tableau 2D ; `df["failure_7d"]` sélectionne une colonne et produit une **Series**, une suite 1D de réponses. Les lignes de `X` et `y` doivent décrire exactement les mêmes observations, dans le même ordre.


In [ ]:
features = [  # Noms des mesures disponibles à l'instant de la décision.
    "temperature_c", "vibration_mm_s", "pressure_bar",
    "load_pct", "age_years", "days_since_maintenance",
]
X = df[features]  # Entrées : une ligne par observation, 6 colonnes par ligne.
y = df["failure_7d"]  # Réponses connues : 0 = pas de panne ; 1 = panne dans 7 jours.
print("X :", X.shape, "| y :", y.shape)  # shape donne les dimensions, pas les valeurs.


### Découper ensemble les entrées et les réponses

`train_test_split` renvoie quatre objets dans cet ordre : entrées conservées, entrées réservées, réponses conservées, réponses réservées. On écrit quatre noms à gauche de `=` pour les récupérer.

`test_size=0.40` réserve 40 % du total dans un bloc **temporaire**, pas dans le test final. `stratify=y` conserve approximativement la proportion de pannes. `random_state=42` refait le même tirage, pour que chacun puisse comparer ses résultats.


In [ ]:
from sklearn.model_selection import train_test_split  # Outil pour séparer des lignes.

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,  # Réserver 40 % ; les 60 % restants servent à apprendre.
    stratify=y,  # Conserver approximativement le taux de panne dans les deux groupes.
    random_state=42,  # Fixer le tirage pseudo-aléatoire, sans améliorer le modèle.
)


Le bloc temporaire contient 40 % du total. Le partager en deux moitiés donne **20 % de validation et 20 % de test**. Le train sert à apprendre, la validation à comparer les choix ; on réserve le test pour la décision finale.


In [ ]:
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp,  # Découper uniquement le bloc réservé précédemment.
    test_size=0.50,  # La moitié de 40 % donne 20 % du total pour le test final.
    stratify=y_temp,  # Préserver la proportion de pannes à l'intérieur de ce bloc.
    random_state=42,
)
print("Train / validation / test :", len(X_train), len(X_validation), len(X_test))


**Vérification.** Les trois nombres doivent s'additionner au nombre total de lignes. Pour ces données pédagogiques, chaque ligne concerne une machine différente. Avec plusieurs observations d'une même machine, ou pour prévoir le futur réel, il faudrait aussi séparer les machines ou le temps : une stratification seule ne suffit pas.


### Préparer les candidats sans les entraîner

Un **dictionnaire** associe un nom à un objet : `{"arbre": modèle}`. `candidates.items()` permettra plus loin de prendre successivement chaque nom et son modèle. Nous préparons trois candidats ; aucun n'a encore appris.

`max_depth` est le nombre maximal de niveaux de questions de l'arbre. `n_estimators` est le nombre d'arbres dans la forêt. La standardisation aide la régression logistique ; les arbres utilisent surtout des comparaisons avec des seuils et n'en ont pas besoin ici.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier  # Une succession de questions sur les mesures.
from sklearn.ensemble import RandomForestClassifier  # Plusieurs arbres qui votent ensemble.
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.base import clone  # Créer une copie non entraînée du candidat finalement choisi.

logistique = make_pipeline(StandardScaler(), LogisticRegression(
    max_iter=1000, class_weight="balanced", random_state=42,
))
arbre = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
foret = RandomForestClassifier(
    n_estimators=150, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1,
)
candidates = {"logistique": logistique, "arbre": arbre, "forêt": foret}


### Définir un calcul commun pour tous

Une **fonction** reçoit des entrées puis `return` renvoie un résultat. Celle-ci reçoit les réponses réelles et les réponses prédites ; elle renvoie les mêmes quatre indicateurs pour tous les candidats. Garder ce calcul identique rend la comparaison honnête.


In [ ]:
def evaluate(truth, predictions):
    """Compter les alertes et mesurer leur qualité face aux réponses connues."""
    return {
        "precision": precision_score(truth, predictions, zero_division=0),
        "recall": recall_score(truth, predictions, zero_division=0),
        "f1": f1_score(truth, predictions, zero_division=0),
        "alertes": int(predictions.sum()),  # Les réponses 1 sont les alertes.
    }


## Étape 1 — entraîner et comparer sur la validation

Dans la boucle ci-dessous, complétez les deux lignes commentées :

1. faire apprendre `candidate` avec **X_train et y_train** ;
2. lui faire prédire **X_validation** et mettre le résultat dans `validation_pred`.

`items()` donne un nom et un modèle à chaque tour. `append` ajoute sa ligne de scores. `**scores` insère les quatre indicateurs dans le dictionnaire de cette ligne : c'est seulement un raccourci pour recopier ses colonnes.


In [ ]:
# Solution guidée
validation_rows = []
for name, candidate in candidates.items():  # Entraîner chaque famille sur le même train.
    candidate.fit(X_train, y_train)
    validation_pred = candidate.predict(X_validation)  # Le test ne participe pas au choix.
    scores = evaluate(y_validation, validation_pred)
    validation_rows.append({"modèle": name, **scores})

validation_results = pd.DataFrame(validation_rows).set_index("modèle")
display(validation_results.round(3))


**Pourquoi cette réponse ?** On compare les familles sur les mêmes lignes et les mêmes réponses de validation. Un meilleur F1 ne veut pas dire moins d'alertes ni absence de pannes manquées. Il faut lire ensemble précision, rappel et nombre d'alertes.


## Lire la table de comparaison

Chaque ligne correspond à un candidat entraîné sur exactement `X_train`. Les prédictions sont ensuite produites sur `X_validation`.

- `precision` mesure la qualité des alertes ;
- `recall` mesure la proportion de pannes retrouvées ;
- `f1` résume ces deux dimensions ;
- `alertes` rend visible la charge opérationnelle.

Le modèle choisi ici maximise F1 pour fournir une règle reproductible. Une entreprise pourrait retenir un autre compromis si le coût des pannes manquées ou des inspections inutiles l’exige.


## Étape 2 — choisir et justifier

Renseignez le nom exact du candidat choisi **dans la table de validation**. Pour cet exercice, la règle proposée est le meilleur F1 ; regardez aussi son rappel et son nombre d'alertes. `loc[nom]` sélectionne la ligne portant ce nom.

Écrivez ensuite une raison : quelle amélioration observe-t-on et quel compromis reste visible ? Il n'est pas encore temps de regarder le test.


In [ ]:
# Solution guidée
selected_name = validation_results["f1"].idxmax()  # Nom de la ligne au F1 maximal.
raison_selection = "Ce candidat a le meilleur F1 de validation parmi les trois familles testées."
display(validation_results.loc[selected_name].round(3))
print("Choix :", selected_name, "|", raison_selection)


**Pourquoi cette réponse ?** `idxmax` renvoie le nom de la ligne ayant le plus grand F1. On garde ce choix avant d'ouvrir le test ; ce protocole ne recherche pas simultanément un seuil personnalisé. Tous les candidats utilisent ici leur décision 0/1 par défaut.


## Choisir ne signifie pas chercher le score le plus grand partout

Une équipe peut préférer un rappel élevé si manquer une panne coûte très cher, ou une meilleure précision si chaque alerte déclenche une intervention lourde. Le nombre d'alertes complète donc les métriques.

Une fois le modèle choisi sur la validation, nous le réentraînons sur train + validation, puis nous ouvrons le test une seule fois.


## Pourquoi réentraîner avant le test final ?

Une fois le candidat choisi, la validation n’est plus nécessaire pour comparer. On rassemble donc train et validation afin de fournir davantage d’exemples au modèle retenu, puis on l’entraîne à nouveau depuis le début.

Le test est ouvert une seule fois pour estimer le résultat final. Si ce score nous conduit à changer de modèle ou d’hyperparamètre, le test devient de fait une nouvelle validation et n’est plus un contrôle indépendant.

Un résultat final doit toujours être accompagné du protocole, du volume de données, des métriques et des limites observées.


### Préparer un nouvel entraînement final

`pd.concat` rassemble les lignes du train et de la validation, dans le même ordre pour X et y. `clone` reprend les réglages du candidat choisi mais crée un modèle **non entraîné**. Il peut alors apprendre sur ce groupe plus grand. Ici, on garde le seuil par défaut : si un seuil personnalisé avait été réglé, son comportement après réentraînement devrait être contrôlé sans utiliser le test.


## Étape 3 — autoriser le contrôle final après le choix

Une fois votre modèle choisi et votre raison écrite, remplacez `False` par `True`. Ce **booléen** indique vrai ou faux. La cellule suivante contrôlera ce choix ; elle ne doit pas vous servir à en chercher un autre.

Avant d'autoriser : où ce modèle a-t-il été choisi ? Les résultats du test ont-ils participé à ce choix ?


In [ ]:
# Solution guidée
autoriser_test = True  # Le modèle est déjà choisi et justifié sur la validation.
print("Contrôle final autorisé :", autoriser_test)


**Pourquoi cette réponse ?** Le booléen rend explicite le moment où on ouvre le test. Le test n'est pas physiquement verrouillé par Python ; c'est notre protocole de travail qui garantit son indépendance.


In [ ]:
selected_model = None  # Le résultat final n'existe qu'après une sélection complète.
test_results = None
if not autoriser_test or validation_results is None or selected_name not in candidates:
    print("Terminez et justifiez la sélection avant d'ouvrir le test.")
else:
    X_train_final = pd.concat([X_train, X_validation])  # Rassembler les entrées connues.
    y_train_final = pd.concat([y_train, y_validation])  # Même ordre pour les réponses.
    selected_model = clone(candidates[selected_name])  # Nouvel objet, mêmes réglages figés.
    selected_model.fit(X_train_final, y_train_final)  # Réapprendre avec plus d'exemples.
    test_predictions = selected_model.predict(X_test)  # Premier contrôle sur ce test.
    test_results = pd.Series(evaluate(y_test, test_predictions), name="test final")
    print("Modèle choisi :", selected_name)
    display(test_results.round(3))


## Étape 4 — expliquer la recommandation au métier

Rédigez une force appuyée par un résultat et une limite réelle. Évitez « le modèle est bon » : dites par exemple combien de pannes il retrouve et combien d'alertes il produit. Distinguez le score observé dans ce cours de ce qu'il faudrait vérifier sur des machines réelles.


In [ ]:
# Solution guidée
force = (
    f"Le modèle {selected_name}, choisi sur validation, retrouve "
    f"{test_results['recall']:.0%} des pannes du test avec "
    f"{int(test_results['alertes'])} alertes sur {len(X_test)} observations."
)
limite = "Le fichier est synthétique ; il faut vérifier ce résultat sur de vraies machines et définir le coût des erreurs."
print("Force :", force)
print("Limite :", limite)


**Pourquoi cette réponse ?** La force cite le rappel et le volume d'alertes plutôt qu'une accuracy isolée. La limite rappelle que le cours teste une méthode sur des données synthétiques, pas la fiabilité d'un détecteur industriel en production.


## Approfondissement accompagné — pourquoi limiter la profondeur ?

Le mini-projet principal est terminé. La suite est une observation guidée, à faire si le temps permet de discuter les résultats. Elle ne change pas votre choix final et n'utilise toujours que le train et la validation.


## Complexité, sous-apprentissage et surapprentissage

![Équilibre entre sous-apprentissage, bonne généralisation et surapprentissage](../../ressources/illustrations/jour_01/04_complexite_generalisation.png)

Un modèle trop simple peut manquer des relations utiles : c'est le **sous-apprentissage**. Un modèle trop complexe peut mémoriser les particularités du jeu d'entraînement : c'est le **surapprentissage**.

Ce que nous recherchons est la **généralisation** : de bonnes prédictions sur des exemples jamais vus. Pour la mesurer honnêtement, on compare les scores d'entraînement et de validation.


La boucle crée un nouvel arbre pour chaque profondeur. `None` signifie ici **aucune limite de profondeur**. Une liste garde les résultats ; chaque `append` ajoute les scores du train et de la validation, pour comparer apprentissage et généralisation.


In [ ]:
profondeurs = [1, 2, 3, 4, 6, 10, None]  # Complexité croissante des règles possibles.
lignes_complexite = []
for profondeur in profondeurs:
    arbre_test = DecisionTreeClassifier(
        max_depth=profondeur, class_weight="balanced", random_state=42,
    )
    arbre_test.fit(X_train, y_train)  # Apprendre avec les mêmes lignes à chaque tour.
    lignes_complexite.append({
        "profondeur": "sans limite" if profondeur is None else str(profondeur),
        "F1 entraînement": f1_score(y_train, arbre_test.predict(X_train), zero_division=0),
        "F1 validation": f1_score(y_validation, arbre_test.predict(X_validation), zero_division=0),
    })
complexite = pd.DataFrame(lignes_complexite)
display(complexite.round(3))


Chaque point horizontal correspond à une profondeur testée. `np.arange` crée les positions `0, 1, 2...` ; `xticks` leur associe les noms lisibles. Deux courbes permettent de voir un écart que le score du train seul cacherait.


In [ ]:
positions = np.arange(len(complexite))  # Une position par réglage testé.
plt.figure(figsize=(8, 4))
plt.plot(positions, complexite["F1 entraînement"], marker="o", label="entraînement")
plt.plot(positions, complexite["F1 validation"], marker="o", label="validation")
plt.xticks(positions, complexite["profondeur"])  # Afficher les profondeurs plutôt que les positions.
plt.xlabel("Profondeur maximale de l'arbre")
plt.ylabel("Score F1")
plt.title("La performance d'entraînement ne suffit pas")
plt.ylim(0, 1.02)  # Les scores F1 sont compris entre 0 et 1.
plt.legend()
plt.show()


### Lire les deux courbes de complexité

Le score d’entraînement augmente généralement avec la profondeur : un arbre plus grand possède davantage de possibilités pour reproduire les exemples vus.

Le score de validation est le véritable contrôle. S’il reste faible avec un arbre très simple, le modèle sous-apprend. S’il baisse alors que le train continue de progresser, l’écart entre les deux courbes signale un surapprentissage probable.

La meilleure profondeur n’est pas forcément celle qui produit le maximum absolu sur un seul découpage. On recherche une zone où la validation reste bonne et où le modèle n’est pas inutilement complexe.


**À discuter avant la suite :** si le F1 du train vaut 1 mais celui de validation est faible, a-t-on appris une règle utile sur de nouvelles machines ou surtout les détails des lignes connues ? Un score de train parfait ne répond pas à cette question tout seul.


### Hyperparamètre et validation croisée

`max_depth` est un **hyperparamètre** : une décision prise avant l'entraînement qui contrôle ici la complexité de l'arbre. Les poids d'un réseau ou les séparations d'un arbre sont au contraire des **paramètres appris** à partir des données.

Une seule validation peut dépendre d'un découpage chanceux. La **validation croisée** recommence l'entraînement sur plusieurs découpages et moyenne les résultats. Des outils comme `GridSearchCV` testent une liste de réglages ; `RandomizedSearchCV` en essaie un échantillon. Leur rôle est d'organiser la comparaison, pas de remplacer le raisonnement métier ni le jeu de test final.


![Principe de la validation croisée](../../ressources/illustrations/jour_01/04_validation_croisee.png)

Dans une validation croisée à quatre plis, les données disponibles sont divisées en quatre blocs. Le modèle est entraîné quatre fois : à chaque tour, trois blocs servent au train et le dernier à la validation. On moyenne ensuite les quatre scores.

Cela donne une estimation moins dépendante d’un seul tirage, mais coûte davantage de temps car l’entraînement est répété. Toutes les étapes de préparation doivent rester à l’intérieur de chaque pli, ce que facilite une Pipeline.

Le test final ne fait pas partie de ces plis. Il reste à l’écart pendant le choix de la famille de modèle et des hyperparamètres.


## Bilan du Jour 1

Vous savez maintenant traduire une question en `X` et `y`, découper les données, entraîner avec `fit`, produire des résultats avec `predict` et lire les erreurs au-delà de l'accuracy.

Demain, nous conserverons exactement ces réflexes. Le réseau de neurones changera la manière d'apprendre les représentations, mais pas la nécessité d'un protocole honnête.
